# Notebook 10 – 1. Input IDs

## Concept
Input IDs are the integer numbers the tokenizer gives to each token. The model never sees text, only these numbers.

## Why it's needed
Neural networks work on numbers. Input IDs are the bridge between human text and model computation.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

text = "I love this movie"
inputs = tokenizer(text, return_tensors="pt")

print("Tokens   :", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Input IDs:", inputs["input_ids"])

## What this example tells us
Each token (including special tokens like [CLS] and [SEP]) maps to one integer ID from the tokenizer vocabulary.

## AI/ML Example
Text -> tokens -> IDs is the first step of every NLP model, including LLMs.

## Business Example
A support-ticket classifier converts each ticket into input IDs before it can predict the category.

## AI/ML Engineer Use Case
Print input IDs and tokens side by side when debugging wrong predictions.

# Notebook 10 – 2. Attention Mask

## Concept
The attention mask tells the model which positions are real tokens (1) and which are padding (0).

## Why it's needed
Texts in a batch have different lengths. Shorter ones are padded, and the model must ignore the padding.

In [ ]:
batch_texts = ["I love this movie", "This is the worst film I have ever seen in my life"]
batch_inputs = tokenizer(batch_texts, padding=True, return_tensors="pt")

print("Input IDs:\n", batch_inputs["input_ids"])
print("Attention mask:\n", batch_inputs["attention_mask"])

## What this example tells us
The shorter sentence is padded with 0 IDs, and its attention mask shows 0 at those padded positions.

## AI/ML Example
Without the mask, padding tokens would influence the representation of the sentence.

## Business Example
A chatbot processes many customer messages of different lengths in one batch for speed.

## AI/ML Engineer Use Case
Always pass the attention mask to the model; never drop it when batching.

# Notebook 10 – 3. Token Type IDs

## Concept
Token type IDs mark which sentence each token belongs to (0 = first sentence, 1 = second). Used by BERT-style models for sentence pairs. DistilBERT does not use them.

## Why it's needed
Tasks like question answering and sentence-pair classification need the model to know where one text ends and the next begins.

In [ ]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
pair = bert_tok("What is AI?", "AI is machine intelligence.", return_tensors="pt")

print("Tokens         :", bert_tok.convert_ids_to_tokens(pair["input_ids"][0]))
print("Token type IDs :", pair["token_type_ids"])
print("DistilBERT keys:", list(inputs.keys()))

## What this example tells us
BERT gives 0 for the first sentence and 1 for the second. The DistilBERT inputs have no token_type_ids key.

## AI/ML Example
Question answering: question = segment 0, context passage = segment 1.

## Business Example
A FAQ search tool compares a user question with a stored answer as a sentence pair.

## AI/ML Engineer Use Case
Check tokenizer output keys first; pass only what the model supports.

# Notebook 10 – 4. Tensor Shape, Batch Dimension and Sequence Dimension

## Concept
Model inputs are tensors of shape (batch_size, sequence_length). Dimension 0 is the batch (how many texts). Dimension 1 is the sequence (how many tokens per text).

## Why it's needed
Shape mismatches are the most common error in model code. Knowing the shape lets you debug quickly.

In [ ]:
print("Single text shape:", inputs["input_ids"].shape)
print("Batch shape      :", batch_inputs["input_ids"].shape)

batch_size, seq_len = batch_inputs["input_ids"].shape
print("Batch dimension (dim 0)   :", batch_size)
print("Sequence dimension (dim 1):", seq_len)

## What this example tells us
Even one text has a batch dimension of 1. The sequence length equals the longest text in the batch after padding.

## AI/ML Example
Models always expect batched input, so a single sentence is shaped (1, tokens).

## Business Example
A batch of 32 customer reviews becomes a tensor (32, longest_review_length).

## AI/ML Engineer Use Case
Print shapes at every step of a pipeline; longer sequences and larger batches also use more memory.

# Notebook 10 – 5. Model Output

## Concept
The model returns an output object containing logits and, when requested, hidden states.

## Why it's needed
Different tasks and options give different outputs. You need to know what to read from them.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

print("Output type  :", type(outputs).__name__)
print("Output keys  :", list(outputs.keys()))

## What this example tells us
The output is an object with named fields, not just a number or a label.

## AI/ML Example
A classification model returns scores per class, not the class name.

## Business Example
A review-sentiment service reads the logits field from the output and ignores the rest.

## AI/ML Engineer Use Case
Inspect outputs.keys() when switching to a new model or task.

# Notebook 10 – 6. Logits

## Concept
Logits are the raw, unnormalized scores the model gives to each class. They can be negative or greater than 1 and do not sum to 1.

## Why it's needed
They are the model's direct output. Probabilities and predictions are both computed from them.

In [ ]:
logits = outputs.logits
print("Logits shape:", logits.shape)
print("Logits      :", logits)
print("Label map   :", model.config.id2label)

## What this example tells us
Shape (1, 2): one text, two classes. The larger logit is the class the model prefers, but the values are not probabilities.

## AI/ML Example
Training loss functions such as cross-entropy work directly on logits.

## Business Example
Raw scores from a fraud model are compared across transactions to rank risk.

## AI/ML Engineer Use Case
Keep logits for training, calibration, and ranking; convert to probabilities only for reporting.

# Notebook 10 – 7. Hidden States

## Concept
Hidden states are the internal vector representations of each token after each layer of the model.

## Why it's needed
They show what the model has learned internally and are used for embeddings, feature extraction, and interpretability.

In [ ]:
hidden_states = outputs.hidden_states
print("Number of hidden state tensors:", len(hidden_states))
print("Shape of last hidden state    :", hidden_states[-1].shape)
print("Meaning: (batch, sequence_length, hidden_size)")

## What this example tells us
There is one tensor for the embeddings plus one per layer. Each token gets a vector of size hidden_size.

## AI/ML Example
The first token's last hidden state is often used as the whole-sentence representation.

## Business Example
Semantic search turns product descriptions into vectors from hidden states to find similar items.

## AI/ML Engineer Use Case
Extract hidden states for embeddings, clustering, or probing what layers learn.

# Notebook 10 – 8. Output Probabilities

## Concept
Probabilities come from applying softmax to the logits. Each value is between 0 and 1, and they add up to 1.

## Why it's needed
Humans need interpretable confidence values, such as 98% positive.

In [ ]:
probs = torch.softmax(logits, dim=-1)
print("Probabilities:", probs)
print("Sum           :", probs.sum().item())

for i, p in enumerate(probs[0]):
    print(f"{model.config.id2label[i]}: {p.item():.4f}")

## What this example tells us
Softmax turns the raw scores into a distribution over the classes. The sum is 1.

## AI/ML Example
The probability is used as a confidence score for the predicted label.

## Business Example
A support system sends low-confidence tickets to a human agent.

## AI/ML Engineer Use Case
Set confidence thresholds on probabilities; remember a high probability does not guarantee a correct answer.

# Notebook 10 – 9. Logits vs Probability vs Prediction

## Concept
Logits are raw scores. Probability is the softmax of the logits. Prediction is the final class chosen, normally with argmax.

## Why it's needed
They are three different stages of the same output. Mixing them up leads to wrong thresholds and wrong conclusions.

In [ ]:
texts = ["I love this movie", "This movie was terrible"]
enc = tokenizer(texts, padding=True, return_tensors="pt")

with torch.no_grad():
    out = model(**enc)

logits = out.logits
probs = torch.softmax(logits, dim=-1)
pred_ids = torch.argmax(probs, dim=-1)

for i, t in enumerate(texts):
    print("Text       :", t)
    print("Logits     :", logits[i].tolist())
    print("Probability:", [round(x, 4) for x in probs[i].tolist()])
    print("Prediction :", model.config.id2label[pred_ids[i].item()])
    print("-" * 40)

## What this example tells us
For each text: logits (raw), probability (normalized), prediction (label). The same argmax comes from logits or probabilities.

## AI/ML Example
Logits feed the loss function, probabilities give confidence, the prediction is what the user sees.

## Business Example
A review dashboard shows the predicted sentiment and its confidence for each customer review.

## AI/ML Engineer Use Case
Log all three during evaluation so you can analyze errors and tune thresholds.